#### Questions:

1) We processed all the files in the metadata and extarcted summary statistics are 
 
   ```
   Found 358 .txt.metadata.json files

    ================================================================================
    SUMMARY STATISTICS
    ================================================================================
    Total unique root_headings: 349
    Total page_h1 categories: 96

At the moment we created "kb_catalog.json"  with 349 entries under "all headings". This will be of file size of 30-50KB(extremly small). 
 - Instead of feeding too many examples in the SYSTEM QUERY PROMPT we use "kb_catalog.json" to feed in the code.
 -  System prompt ≠ full KB
    
    The model only needs a few examples of classifications.

    All matching is done offline via code.
    
- This prevents model to hallucinate and code valides only real heading, fuzzy expanded

2) In the next version do we want generate representative prompt sections(may be 5-8 per category) from the root_heading_extarcted?
    - Process all 349 and auto cluster them to X(may be 10-12) semantic groups?


In [ ]:
import os
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
# Generate root heading catalog by running the helper script
# Pass the metadata directory and specify output file
!python ../helpers/apug/root_heading_catalog.py --metadata_dir ../data --output ../data/kb_catalog.json

In [ ]:
# This script extracts unique root_headings from metadata JSON files
# and groups them by the actual page_h1 found in the metadata file.
# This is useful for generating a catalog of root_headings for validation
# and understanding their distribution across different page headings.
# Only used for internal analysis and catalog generation.
# Then we have written the script root_heading_catalog.py in helpers/apug/root_heading_catalog.py which does the same in a better way.
# --------------------------------------------------------------------------- #
# Don't edit below this line                                                  #
# --------------------------------------------------------------------------- #

# Import necessary modules
import json
from pathlib import Path
from collections import defaultdict

# Function for normalizing page_h1 values
def normalize_page_h1(value: str) -> str:
    """
    Optional: Normalize page_h1 for cleaner grouping.
    - Trim whitespace
    - Replace empty or missing headings with "Unknown Page"
    """
    if not value or not value.strip():
        return "Unknown Page"
    return value.strip()

# Main extraction function
def extract_root_headings(metadata_dir):
    """
    Extract unique root_headings from .txt.metadata.json files.
    Group them by the actual page_h1 found in the metadata file
    (rather than using predefined category rules).
    """

    headings_by_page = defaultdict(set)
    all_headings = set()

    files_processed = 0
    files_without_heading = 0
    files_with_errors = 0

    # Find *.txt.metadata.json files recursively
    json_files = list(Path(metadata_dir).glob("**/*.txt.metadata.json"))
    print(f"Found {len(json_files)} .txt.metadata.json files\n")

    if not json_files:
        print(" No metadata files found!")
        print(f"Searched in: {Path(metadata_dir).absolute()}")
        return {}, set()

    # Process each file
    for json_file in json_files:
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            metadata = data.get("metadataAttributes", {})

            page_h1 = normalize_page_h1(metadata.get("page_h1", ""))
            root_heading = metadata.get("root_heading", "").strip()

            # Skip files without root_heading
            if not root_heading:
                files_without_heading += 1
                continue

            # Add to global set
            all_headings.add(root_heading)

            # Add to category bucket based on page_h1
            headings_by_page[page_h1].add(root_heading)

            files_processed += 1

        except Exception as e:
            files_with_errors += 1
            print(f" Error reading {json_file.name}: {e}")

    # Summary stats
    print(f" Files with root_heading: {files_processed}/{len(json_files)}")
    print(f" Files without root_heading: {files_without_heading}")
    if files_with_errors > 0:
        print(f" Errors encountered in {files_with_errors} files")
    print()

    if not all_headings:
        print("No root_heading values found in any metadata files.")
        return {}, set()

    # Print formatted report to console
    print("\n" + "=" * 80)
    print("AVAILABLE ROOT_HEADINGS (Grouped by page_h1)")
    print("=" * 80 + "\n")

    for page, headings in sorted(headings_by_page.items()):
        print(f"{page}:")
        for h in sorted(headings):
            print(f'  - "{h}"')
        print()

    print("=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Total unique root_headings: {len(all_headings)}")
    print(f"Total page_h1 categories: {len(headings_by_page)}\n")

    print("All Unique Root Headings (alphabetical):")
    for heading in sorted(all_headings):
        print(f'  - "{heading}"')

    # Generate array version for validation code
    print("\n" + "=" * 80)
    print("VALIDATION LIST (copy/paste)")
    print("=" * 80)
    print("VALID_ROOT_HEADINGS = [")
    for heading in sorted(all_headings):
        print(f'    "{heading}",')
    print("]\n")

    return headings_by_page, all_headings


# --------------------------------------------------------------------------- #
# Entry point                                                                 #
# --------------------------------------------------------------------------- #

if __name__ == "__main__":
    METADATA_DIR = "../data"  # default path

    # Validate input directory
    if not Path(METADATA_DIR).exists():
        print(f" Directory not found: {METADATA_DIR}")
        print(f" Absolute path: {Path(METADATA_DIR).absolute()}")
        METADATA_DIR = input("\nEnter metadata directory path: ").strip()

    headings_by_page, all_headings = extract_root_headings(METADATA_DIR)

    # Write output to a file
    if all_headings:
        with open("root_headings_output.txt", "w", encoding="utf-8") as f:
            f.write("=" * 80 + "\n")
            f.write("ROOT_HEADINGS BY page_h1\n")
            f.write("=" * 80 + "\n\n")

            for page_h1, headings in sorted(headings_by_page.items()):
                f.write(f"{page_h1}:\n")
                for h in sorted(headings):
                    f.write(f'  - "{h}"\n')
                f.write("\n")

            f.write("\n" + "=" * 80 + "\n")
            f.write("All Unique Root Headings\n")
            f.write("=" * 80 + "\n")
            for h in sorted(all_headings):
                f.write(f'- "{h}"\n')

        print("\n Output written to: root_headings_output.txt")
    else:
        print("\n No output generated — no root_heading values found.")


#

### How we are going to implement filter genrator?

STEP 1: Create a Query Analyzer Layer (query_analyzer.py)
that sits BEFORE existing ask() / chat() functions
This analyzer function examines the user's query and extracts intent, complexity, and scope

---

STEP 2: Create a Filter Generator (filter_generator.py)

This takes the analyzer's output and converts it into your existing
MetadataFilters format
Crucially: It outputs the SAME filter structure as current system already uses

Validation checkpoint:

Generate filters for test queries

Compare against what you would have manually written Test that generated filters actually work with your current ask()function

---

STEP 3: Create a Wrapper Function 

Create a NEW function ask_smart()
that wraps your existing
ask() function
This is where Steps 1 and 2 come together
How it works:

ask_smart(query, auto_filter=True):
    if auto_filter:
        analysis = query_analyzer.analyze(query)
        filters = filter_generator.generate(analysis)
    else:
        filters = None  # or user-provided filters
    
    return ask(query, metadata_filters=filters)


Validation checkpoint:

Run parallel tests: same query through
ask()
(manual filters) vs
ask_smart()
(auto filters)
Compare: retrieval results, answer quality, speed

---

STEP 4: Add Strategy Selection Logic
What to do:

Enhance your
query_analyzer.py
to also output a retrieval strategy recommendation
Based on query complexity, decide: filtered, broad, or hybrid approach
How it works:

Simple, clear query with obvious metadata → FILTERED (use generated filters)
Ambiguous, complex query → BROAD (no filters, higher top_k)
Medium complexity → HYBRID (light filters + higher top_k)
Implementation in your system:

Strategy = FILTERED:
    → Use generated filters + your default NUMBER_OF_RESULTS

Strategy = BROAD:
    → No filters + increase NUMBER_OF_RESULTS to 15-20

Strategy = HYBRID:
    → Use only high-confidence filters + increase NUMBER_OF_RESULTS to 10-15
Why this matters:

Some queries NEED broad search (exploratory, unclear scope)
Some queries BENEFIT from tight filtering (specific, narrow scope)
This decision logic is your Phase 1 from the flowcharts
Validation checkpoint:

Manually label 50 queries as needing "filtered", "broad", or "hybrid"
Check if your strategy selector agrees with your manual labels
Test actual retrieval: does strategy choice improve results?

---

STEP 5: Add a Quality Assessment Module (Post-Generation)
What to do:

Create
quality_assessor.py
that runs AFTER you get an answer from Bedrock
This module evaluates the answer quality before showing it to the user
How it works:

Takes: generated answer + retrieved citations
Outputs: quality scores (coverage, consistency, source diversity, grounding)
Decision: Is quality acceptable? If not, what's the issue?
Implementation approach:

answer, citations = chat(query, filters)
quality = quality_assessor.assess(query, answer, citations)

if quality.is_good():
    return answer  # Show to user
else:
    # Either auto-retry with different strategy OR inform user
    return refinement_suggestions
Why this matters:

Catches low-quality answers before user sees them
Identifies when filters were too restrictive (no/few results)
Identifies when answer doesn't match query intent
This is your Phase 3 from the flowcharts
Validation checkpoint:

Compare quality scores against your own human judgment
Test: Does it correctly flag bad answers?
Test: Does it correctly pass good answers?

---

STEP 6: Implement the Retry/Refinement Loop
What to do:

When quality assessment fails, automatically try an alternative strategy
Example: If FILTERED returns poor results → retry with BROAD
How it works:

Strategy attempt 1: FILTERED
    ↓ (quality check fails: too few sources)
Strategy attempt 2: BROAD
    ↓ (quality check passes)
Return answer from attempt 2
Implementation considerations:

Max retries: 2 (avoid infinite loops)
Each retry uses a DIFFERENT strategy
Log each attempt for analysis
Show user which strategy worked (transparency)
Why this matters:

Automatic recovery from over-filtering
Better user experience (no manual retry needed)
You learn which strategies work for which query types


---

STEP 7: Integrate with Your Query Evaluator
What to do:

Enhance your query evaluator to show:
Which strategy was used (filtered/broad/hybrid)
What filters were auto-generated
Quality scores
Whether refinement happened
Display enhancements:

Answer: [your existing answer display]

---
Metadata:
Strategy: FILTERED
Auto-generated filters: page_url contains "data-access", section = "Athena"
Quality Score: 0.85/1.0 (Good)
Sources: 5 documents (3 unique pages)
Retrieval time: 1.2s
Why this matters:

Transparency: users understand how answer was generated
Debugging: you can see if filters are working correctly
Trust: users see the system's reasoning
